# Section 1 — Preamble & Environment

This section initialises the environment and logging.

In [ ]:
import os
import json
import platform
from datetime import datetime

import torch

print('Python', platform.python_version())
print('PyTorch', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
run_id = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
print('Run ID:', run_id)
os.makedirs('tb_logs', exist_ok=True)

# Section 2 — JSON Configuration

In [ ]:
{
  "data": {
    "csv_path": "./diabetic_data.csv",
    "id_mapping_path": "./IDS_mapping.csv",
    "identifier_cols": { "encounter_id": "encounter_id", "patient_id": "patient_nbr" },
    "target": { "name": "readmitted", "positive_values": ["<30"] },
    "filters": {
      "min_los": 1,
      "max_los": 14,
      "exclude_discharge_to_ids": [11, 13, 14, 19, 20, 21],
      "first_encounter_per_patient": true
    },
    "columns": {
      "numeric": [
        "time_in_hospital","num_lab_procedures","num_procedures",
        "num_medications","number_outpatient","number_emergency",
        "number_inpatient","number_diagnoses"
      ],
      "categorical_low_card": [
        "gender","race","A1Cresult","max_glu_serum","diabetesMed","change"
      ],
      "icd_cols": ["diag_1","diag_2","diag_3"],
      "drug_cols": [
        "metformin","repaglinide","nateglinide","chlorpropamide","glimepiride",
        "acetohexamide","glipizide","glyburide","tolbutamide","pioglitazone",
        "rosiglitazone","acarbose","miglitol","troglitazone","tolazamide",
        "examide","sitagliptin","insulin","glyburide-metformin",
        "glipizide-metformin","glimepiride-pioglitazone","metformin-rosiglitazone",
        "metformin-pioglitazone"
      ],
      "hospital_col": "hospital_id",
      "specialty_col": "medical_specialty",
      "admission_type_col": "admission_type_id",
      "admission_source_col": "admission_source_id",
      "discharge_disposition_col": "discharge_disposition_id"
    },
    "preprocessing": {
      "numeric_imputer": "median",
      "categorical_imputer": "most_frequent",
      "scaler": "standard",
      "categorical_handling": "embedding",
      "use_unknown_category": true,
      "unknown_label": "UNKNOWN",
      "min_freq_for_category": 20,
      "truncate_icd_to_3_digits": true,
      "map_icd_to_group": true,
      "map_drug_to_class": true
    },
    "splits": {
      "strategy": "group_k_fold",
      "group_by": "patient",
      "n_splits": 5,
      "seed": 42,
      "stratify_by_target": true
    }
  },
  "graph": {
    "node_types_enabled": {
      "encounter": true, "icd": true, "icd_group": true, "drug": true, "drug_class": true,
      "hosp": true, "specialty": true, "admission_source": true, "admission_type": true,
      "discharge_disposition": true
    },
    "edge_types_enabled": {
      "encounter__has_icd__icd": true,
      "icd__is_a__icd_group": true,
      "encounter__has_drug__drug": true,
      "drug__belongs_to__drug_class": true,
      "encounter__at_hospital__hosp": true,
      "encounter__has_specialty__specialty": true,
      "encounter__has_admission_source__admission_source": true,
      "encounter__has_admission_type__admission_type": true,
      "encounter__has_discharge__discharge_disposition": true,
      "reverse_edges": true
    },
    "edge_featureing": {
      "has_drug": { "relation_subtypes_by_status": true, "edge_attr_status": false }
    },
    "feature_dims": {
      "encounter_tabular_proj_dim": 64,
      "embedding_dims": {
        "icd": 64, "icd_group": 32, "drug": 64, "drug_class": 32,
        "hosp": 16, "specialty": 16, "admission_source": 8,
        "admission_type": 8, "discharge_disposition": 8
      }
    },
    "oov_nodes": true,
    "artifacts_dir": "./artifacts"
  },
  "model": {
    "arch": "HGT",
    "hidden_dim": 128,
    "num_layers": 3,
    "dropout": 0.3,
    "heads": 4,
    "rgcn_bases": 30,
    "act": "gelu",
    "norm": "layer",
    "use_edge_attr_for_drug_status": false,
    "loss": { "type": "bce_with_logits", "pos_weight": "auto" }
  },
  "train": {
    "epochs": 100,
    "early_stopping_patience": 15,
    "early_stopping_metric": "auprc",
    "optimizer": { "name": "AdamW", "lr": 0.002, "weight_decay": 0.0001 },
    "scheduler": { "name": "cosine", "warmup_epochs": 5 },
    "batching": {
      "batch_size_encounters": 1024,
      "fanouts_per_layer_by_relation": {
        "encounter__has_icd__icd": [15,10,5],
        "encounter__has_drug__drug": [15,10,5],
        "encounter__at_hospital__hosp": [5,5,5],
        "encounter__has_specialty__specialty": [5,5,5],
        "encounter__has_admission_source__admission_source": [5,5,5],
        "encounter__has_admission_type__admission_type": [5,5,5],
        "encounter__has_discharge__discharge_disposition": [5,5,5],
        "icd__is_a__icd_group": [3,3,3]
      }
    },
    "val_every": 1,
    "gradient_clip_norm": 1.0,
    "deterministic": true,
    "seed": 42,
    "tensorboard_logdir": "./tb_logs",
    "save_best_on": "auprc"
  },
  "evaluation": {
    "metrics_primary": ["auprc","auroc"],
    "metrics_secondary": ["f1_pos","precision_pos","recall_pos","balanced_accuracy","brier","ece"],
    "threshold_tuning": {
      "optimize_for": "f1_pos",
      "grid": [0.1,0.2,0.3,0.4,0.5,0.6],
      "calibration": "isotonic"
    },
    "plots": { "roc": true, "pr": true, "calibration": true, "confusion": true, "decision_curves": true },
    "subgroup_metrics": ["age","race","gender","hospital_id"]
  },
  "inference": {
    "inductive": true,
    "build_star_subgraph_on_the_fly": true,
    "oov_handling": "UNKNOWN",
    "batch_predict_csv_path": null,
    "output_predictions_path": "./predictions.csv"
  },
  "baseline": {
    "tabular_mlp": {
      "enabled": true,
      "hidden_dims": [256,128],
      "dropout": 0.2,
      "epochs": 60,
      "batch_size": 2048,
      "optimizer": { "name": "AdamW", "lr": 0.001, "weight_decay": 0.0001 }
    },
    "xgboost": { "enabled": false }
  }
}

# Section 3 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 4 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 5 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 6 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 7 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 8 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 9 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 10 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 11 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 12 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 13 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 14 — Placeholder

In [ ]:
# TODO: implement section logic

# Section 15 — Placeholder

In [ ]:
# TODO: implement section logic